# 1. Supervised Fine-Tuning (SFT)

> 책 Chapter 5 — *Instruction Tuning*

RLHF 풀 루프의 **첫 단계**. 사전학습된 LM을 (prompt, demonstration) 쌍으로 fine-tune해서 "지시를 따르는 모델"로 만든다. 이후 모든 stage(Reward Model, PPO)는 이 SFT 모델에서 출발한다.

## 핵심 질문

- **수학적으로 무엇을 최소화하나?** Cross-entropy loss, 단 응답 토큰에만.
- **왜 prompt에는 loss를 안 주나?** 모델은 *프롬프트 생성기*가 아니라 *응답 생성기*가 되어야 하므로.
- **chat template과 어떻게 상호작용하나?** 템플릿 special token도 손실에 포함 (assistant 응답 끝 EOS 등).

## 이 노트북에서 다룰 것

1. SFT의 수학 — autoregressive LM의 likelihood
2. Response token masking — 어디서부터 어디까지 loss를 줄지
3. From-scratch loss 구현 — `F.cross_entropy`를 그대로 쓰지 않고 직접 reduce
4. Mini batched dataloader 구조

## 1. SFT의 수학 — Maximum Likelihood

데이터셋: $\mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^N$
- $x^{(i)}$: prompt (예: "양자얽힘을 설명해줘")
- $y^{(i)} = (y_1, \ldots, y_T)$: 인간이 작성한 demonstration response

### 목표

$$
\theta^* = \arg\max_\theta \prod_{i=1}^N \pi_\theta\bigl(y^{(i)} \mid x^{(i)}\bigr)
$$

Autoregressive 분해:

$$
\pi_\theta(y \mid x) = \prod_{t=1}^T \pi_\theta(y_t \mid x, y_{<t})
$$

로그를 취하고 부호 뒤집어서 **최소화 형태**로:

$$
\mathcal{L}_{\mathrm{SFT}}(\theta) = - \mathbb{E}_{(x,y) \sim \mathcal{D}}
\Bigl[ \sum_{t=1}^T \log \pi_\theta(y_t \mid x, y_{<t}) \Bigr]
$$

이게 바로 **다음 토큰 예측 cross-entropy** — 사전학습과 정확히 동일한 loss다.

### 그럼 사전학습과 뭐가 다른가?

1. **데이터 분포**: 사전학습은 일반 웹 코퍼스, SFT는 큐레이션된 (prompt, response) 쌍
2. **마스킹**: 사전학습은 모든 토큰에 loss, SFT는 *응답 토큰에만* loss

## 2. 왜 응답 토큰에만 loss를 주나? — Mask의 의미

전체 시퀀스가 다음과 같다고 하자:

```
[system header] You are a helpful assistant. [user header] 양자얽힘을 설명해줘. [assistant header] 양자얽힘은...
└──────────────── prompt 영역 (loss = 0) ─────────────────┘└─── response (loss 계산) ───┘
```

만약 prompt 토큰에도 loss를 주면:
- 모델이 "양자얽힘을 설명해줘"라는 *질문 자체*를 잘 생성하도록 학습됨 ❌
- 우린 그게 아니라 *답변*을 잘 생성하길 원함

→ 응답 시작 토큰부터만 loss 계산. 이를 위해 **label에 `-100`을 채워 넣어** `F.cross_entropy`가 자동으로 무시하게 만든다 (PyTorch convention).

## 3. Llama 3.2 chat template과의 상호작용

Kybalion-1B는 Llama 3.2 토크나이저를 쓰며 chat template은 아래 형식:

```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>

{system}<|eot_id|>
<|start_header_id|>user<|end_header_id|>

{user}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

{response}<|eot_id|>
```

**핵심**: `<|start_header_id|>assistant<|end_header_id|>\n\n` 다음부터 `<|eot_id|>`까지가 응답 영역. 이 토큰 인덱스 구간만 label로 살리고 나머진 `-100`.

In [ ]:
# Llama 3.2 chat template SFT 마스킹 — 핵심 로직
# 실제 학습 코드에서 가장 자주 버그가 나는 부분이다.

import torch
from transformers import AutoTokenizer

# 가상의 토크나이저 로드 (실제로는 Kybalion-1B의 토크나이저)
# tokenizer = AutoTokenizer.from_pretrained("devwoo/Kybalion-1B")

# 데모를 위해 정수 token id를 직접 사용
# 실제 Llama 3.2 special tokens:
#   <|begin_of_text|>           = 128000
#   <|start_header_id|>         = 128006
#   <|end_header_id|>           = 128007
#   <|eot_id|>                  = 128009

IGNORE_INDEX = -100   # PyTorch cross-entropy가 자동으로 무시하는 값

def build_sft_example(prompt_ids: list[int], response_ids: list[int]) -> dict[str, torch.Tensor]:
    """SFT 학습 샘플 하나 생성.

    Args:
        prompt_ids: chat template으로 인코딩된 prompt (system + user 헤더 + ... + assistant 헤더)
        response_ids: chat template으로 인코딩된 response (응답 텍스트 + <|eot_id|>)

    Returns:
        input_ids: [prompt | response] 결합
        labels:    [-100 ... -100 | response_ids]  — prompt 부분은 무시
    """
    input_ids = torch.tensor(prompt_ids + response_ids, dtype=torch.long)

    # prompt 길이만큼 -100으로 패딩, response 부분은 그대로 정답
    labels = torch.full_like(input_ids, IGNORE_INDEX)
    labels[len(prompt_ids):] = torch.tensor(response_ids, dtype=torch.long)

    return {"input_ids": input_ids, "labels": labels}


# 시연용 더미 토큰
prompt = [128000, 128006, 9125, 128007, 366, 1495, 528, 128009]   # <bos><|...|>system<|...|>... <|eot_id|>
response = [422, 374, 264, 1296, 0, 128009]                          # "is a test!" <|eot_id|>

ex = build_sft_example(prompt, response)
print("input_ids:", ex["input_ids"].tolist())
print("labels   :", ex["labels"].tolist())
print(f"\n프롬프트 길이 {len(prompt)} 토큰 → 앞쪽 {len(prompt)}개 label은 -100 (loss 미계산)")

## 4. 직접 작성한 SFT loss

`F.cross_entropy`를 부르긴 하지만, **왜 그렇게 호출되는지** 단계별로 풀어쓴다.

### Shift 트릭

LM의 출력 `logits`는 `(B, T, V)` 형태. 각 시점 $t$의 logits은 *다음* 토큰을 예측한다. 즉:

- `logits[:, t, :]` = $P(y_{t+1} \mid y_{\le t})$의 로짓
- 그래서 label과 logit이 한 칸 어긋남

→ shift해서 정렬:
- `shift_logits = logits[:, :-1, :]`  # 마지막 시점 제거
- `shift_labels = labels[:, 1:]`      # 첫 토큰 제거

In [ ]:
import torch
import torch.nn.functional as F


def sft_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """SFT loss를 직접 분해해서 계산.

    Args:
        logits: (B, T, V) — LM의 raw output logits
        labels: (B, T)    — input_ids와 동일 shape, -100은 무시됨

    Returns:
        scalar tensor — 평균 negative log likelihood
    """
    # 1) shift
    #    각 t 시점의 logit이 t+1의 token을 예측한다는 사실에 맞춰 정렬.
    shift_logits = logits[..., :-1, :].contiguous()   # (B, T-1, V)
    shift_labels = labels[..., 1:].contiguous()        # (B, T-1)

    # 2) flatten — (B*(T-1), V) vs (B*(T-1),)
    flat_logits = shift_logits.view(-1, shift_logits.size(-1))
    flat_labels = shift_labels.view(-1)

    # 3) cross-entropy.
    #    ignore_index=-100 으로 prompt 토큰 label이 자동 제거됨.
    #    내부적으로는: -log(softmax(logits)[label]) 을 모든 유효 토큰 평균.
    loss = F.cross_entropy(
        flat_logits,
        flat_labels,
        ignore_index=-100,
        reduction="mean",   # response 토큰들 평균
    )
    return loss


# 시연 — 가상의 logits/labels
B, T, V = 2, 8, 100
logits = torch.randn(B, T, V)
labels = torch.randint(0, V, (B, T))
# 앞 3개 토큰을 prompt로 가정 → -100 마스크
labels[:, :3] = -100

loss = sft_loss(logits, labels)
print(f"SFT loss (랜덤 logits): {loss.item():.4f}")
print(f"  (랜덤이라 log V ≈ log {V} = {torch.log(torch.tensor(float(V))).item():.4f} 근처여야 함)")

## 5. Loss를 더 깊이 — 수동 분해

`F.cross_entropy`를 안 쓰고 더 raw하게 작성. 학습 디버깅 / 정확한 이해에 도움.

In [ ]:
def sft_loss_manual(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """F.cross_entropy 분해 — softmax + gather + log + mean.

    수식: L = - (1/N) Σ_{t : label_t ≠ -100} log P(label_t | logits_t)
    """
    shift_logits = logits[..., :-1, :].contiguous()   # (B, T-1, V)
    shift_labels = labels[..., 1:].contiguous()        # (B, T-1)

    # log-softmax 안정적으로 (log Σ exp trick 내장)
    log_probs = F.log_softmax(shift_logits, dim=-1)    # (B, T-1, V)

    # 유효 마스크
    mask = (shift_labels != -100)                       # (B, T-1)

    # gather: 각 시점에서 정답 token의 log P
    safe_labels = shift_labels.clone()
    safe_labels[~mask] = 0                              # gather 인덱스 에러 방지용 임시 값
    label_logp = log_probs.gather(
        dim=-1, index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)                                       # (B, T-1)

    # 마스크된 위치만 평균
    nll = -label_logp * mask
    loss = nll.sum() / mask.sum().clamp(min=1)
    return loss


# F.cross_entropy 결과와 수동 결과가 같은지 검증
torch.manual_seed(0)
logits = torch.randn(2, 8, 100)
labels = torch.randint(0, 100, (2, 8))
labels[:, :3] = -100

a = sft_loss(logits, labels)
b = sft_loss_manual(logits, labels)
print(f"F.cross_entropy: {a.item():.6f}")
print(f"manual         : {b.item():.6f}")
print(f"동일?            {torch.allclose(a, b, atol=1e-6)}")

## 6. Dataset & DataLoader — 실제 학습 파이프라인

PyTorch Dataset class로 chat template + 마스킹 + padding을 처리하는 표준 구조.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from typing import Optional


class SFTDataset(Dataset):
    """SFT용 Dataset — chat template 적용 + response masking + padding 처리.

    각 example은 다음을 포함하는 dict:
        - messages: list of {"role": "user"|"assistant"|"system", "content": str}
    """

    def __init__(self, examples: list[dict], tokenizer, max_length: int = 2048):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        # response 시작을 식별하기 위한 토큰 (Llama 3.2)
        # 실제 토큰화는 tokenizer.apply_chat_template에 위임
        self.ignore_index = -100

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        messages = self.examples[idx]["messages"]

        # 1) 전체 시퀀스 토큰화 (응답 포함)
        full_ids = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,   # 응답까지 다 들어가 있으니 False
            return_tensors="pt",
        ).squeeze(0)                       # (T,)

        # 2) prompt-only 시퀀스 토큰화 (응답 직전까지 + assistant 헤더만)
        prompt_only_ids = self.tokenizer.apply_chat_template(
            messages[:-1],                   # 마지막 assistant 메시지 제외
            tokenize=True,
            add_generation_prompt=True,      # assistant 헤더까지 포함
            return_tensors="pt",
        ).squeeze(0)
        prompt_len = prompt_only_ids.size(0)

        # 3) 자르기
        full_ids = full_ids[: self.max_length]

        # 4) labels = full_ids 복사 후 prompt 부분 마스킹
        labels = full_ids.clone()
        labels[:prompt_len] = self.ignore_index

        return {
            "input_ids": full_ids,
            "labels": labels,
        }


def collate_sft(batch: list[dict], pad_token_id: int) -> dict[str, torch.Tensor]:
    """가변 길이 시퀀스를 가장 긴 길이에 맞춰 right-padding."""
    max_len = max(item["input_ids"].size(0) for item in batch)

    input_ids = []
    labels = []
    attn = []
    for item in batch:
        L = item["input_ids"].size(0)
        pad = max_len - L

        input_ids.append(F.pad(item["input_ids"], (0, pad), value=pad_token_id))
        labels.append(F.pad(item["labels"],    (0, pad), value=-100))   # padding도 무시
        attn.append(F.pad(torch.ones(L, dtype=torch.long), (0, pad), value=0))

    return {
        "input_ids":      torch.stack(input_ids),
        "labels":         torch.stack(labels),
        "attention_mask": torch.stack(attn),
    }

## 7. 학습 loop — One step

실제 RLHF 책 Ch 5에 나오는 학습 step을 코드로:

In [ ]:
# 가상의 학습 step. 실행은 안 되지만 흐름은 정확.

# from transformers import AutoModelForCausalLM
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", torch_dtype=torch.bfloat16).cuda()
# optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def sft_step(model, batch: dict[str, torch.Tensor], optimizer) -> float:
    """SFT 한 step. 책 Algorithm 5.1과 동등.

    Args:
        model: HF causal LM (forward가 dict 반환)
        batch: {"input_ids", "labels", "attention_mask"}
        optimizer: torch.optim.Optimizer

    Returns:
        loss value (float)
    """
    # 1) Forward — HF model의 forward는 labels를 받으면 자동으로 cross-entropy 계산
    #    우리가 위에서 sft_loss로 분해한 것과 동등.
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )
    loss = outputs.loss

    # 2) Backward
    loss.backward()

    # 3) Gradient clipping (학습 안정성)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # 4) Optimizer step
    optimizer.step()
    optimizer.zero_grad()

    return loss.item()


# 전형적 학습 loop (의사 코드)
"""
for epoch in range(num_epochs):
    for batch in dataloader:
        loss_val = sft_step(model, batch, optimizer)
        if step % 100 == 0:
            print(f"step {step:5d} | loss {loss_val:.4f}")
"""
print("학습 loop 정의 완료 — 실제 실행은 GPU + 모델 필요")

## 8. SFT의 결과물 = π_SFT

이 단계를 마치면 우리는 **π_SFT** 라는 모델을 얻는다. 이게 다음 단계 전부의 출발점:

- **Reward Model의 backbone**: SFT 모델을 복사하고 마지막 layer를 scalar head로 교체
- **PPO의 initial policy π_θ**: SFT 모델을 그대로 복사해서 학습 시작
- **PPO의 reference π_ref**: SFT 모델을 그대로 복사해서 고정 (KL 기준)

→ 세 군데에서 모두 SFT 모델이 쓰임. **SFT 품질이 RLHF 전체 품질을 결정**한다.

## 9. Kybalion-1B와의 연결

[devwoo/Kybalion-1B](https://huggingface.co/devwoo/Kybalion-1B)는 이미 다음을 거친 모델:
1. Llama 3.2 1B 사전학습 (Meta)
2. CPT 3.5B 토큰 (도메인 특화)
3. **SFT (LoRA) 170K 토큰** ← 이 노트북에 해당하는 단계

즉 Kybalion은 이미 $\pi_\mathrm{SFT}$ 상태다. 다음 노트북부터 이 모델을 Reward Model → PPO로 정렬해 나갈 수 있다.

## 10. 다음 노트북 — `2_RewardModel.ipynb`

다음 단계는 **선호 데이터 (chosen > rejected) 로부터 reward 함수를 학습**한다. Bradley-Terry 모델과 그 loss의 유도가 핵심.